# Decentralized AI Cloud — Kaggle Community GPU Worker

This notebook connects your free **Kaggle GPU (Tesla T4 x2 or P100)** to the **Decentralized AI Compute Cloud** platform as an active compute provider.

### Important Settings in Kaggle:
1. **Accelerator:** On the right sidebar, set **Accelerator** to `GPU T4 x2` or `GPU P100`.
2. **Internet:** Set **Internet** to `Internet on` *(required to connect to the coordinator)*.

In [ ]:
# Step 1: Verify NVIDIA GPU Hardware
!nvidia-smi

In [ ]:
# Step 2: Clone Repository & Install Worker Dependencies
!git clone https://github.com/liladhiee-web/Decentralized.git
%cd Decentralized
!pip install -q websockets pynvml httpx pydantic rich

In [ ]:
# Step 3: Configure Coordinator Tunnel URL & Worker Secret Token
import os

# Enter your Coordinator URL (Cloudflare Tunnel or Public VPS URL)
os.environ["COORDINATOR_URL"] = "https://meyer-enclosure-jungle-burton.trycloudflare.com"
os.environ["WORKER_TOKEN"] = "Tyhb0yvipG_Kkz4CKGNSw1jBBWTS0gfrNZ9a6aIUbWM"
os.environ["WORKER_NAME"] = "kaggle-gpu-t4"

# Step 4: Launch Worker Daemon (starts telemetry and task listener)
!python -m worker.agent

### Standalone Worker Runner (No Git Required)
If you prefer running the worker directly in a single notebook cell without cloning the repository, run the cell below:

In [ ]:
import os, sys, asyncio, json, time
!pip install -q websockets pynvml httpx pydantic

import pynvml
import websockets

COORDINATOR_WS = "wss://meyer-enclosure-jungle-burton.trycloudflare.com/ws/workers"
WORKER_TOKEN = "Tyhb0yvipG_Kkz4CKGNSw1jBBWTS0gfrNZ9a6aIUbWM"
WORKER_NAME = "kaggle-nvidia-t4"

# Initialize NVML to query physical Kaggle GPU
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
gpu_name = pynvml.nvmlDeviceGetName(handle)
mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)

print(f"[+] Detected GPU: {gpu_name} ({mem_info.total // (1024*1024)} MB VRAM)")
print(f"[*] Connecting to {COORDINATOR_WS}...")

async def run_worker():
    async with websockets.connect(COORDINATOR_WS) as ws:
        # 1. Send registration handshake
        auth_msg = {
            "type": "worker_auth",
            "token": WORKER_TOKEN,
            "name": WORKER_NAME,
            "gpu_devices": [{
                "gpu_index": 0,
                "name": str(gpu_name),
                "total_vram_mb": mem_info.total // (1024*1024),
                "free_vram_mb": mem_info.free // (1024*1024),
                "cuda_compute_capability": "7.5",
                "driver_version": "535.154.05",
                "cuda_version": "12.2",
                "temperature_celsius": 35.0,
                "power_draw_watts": 25.0,
                "utilization_gpu_percent": 0,
                "utilization_mem_percent": 0,
                "is_available": True
            }]
        }
        await ws.send(json.dumps(auth_msg))
        ack = await ws.recv()
        print(f"[+] Handshake Accepted: {ack}")
        
        # 2. Heartbeat loop (every 15s)
        while True:
            mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
            util = pynvml.nvmlDeviceGetUtilizationRates(handle)
            temp = pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU)
            heartbeat = {
                "type": "worker_heartbeat",
                "token": WORKER_TOKEN,
                "gpu_devices": [{
                    "gpu_index": 0,
                    "name": str(gpu_name),
                    "free_vram_mb": mem.free // (1024*1024),
                    "total_vram_mb": mem.total // (1024*1024),
                    "temperature_celsius": float(temp),
                    "utilization_gpu_percent": util.gpu,
                    "utilization_mem_percent": util.memory,
                    "is_available": True
                }]
            }
            await ws.send(json.dumps(heartbeat))
            print(f"[♥] Telemetry synced: Util: {util.gpu}% | Free VRAM: {mem.free // (1024*1024)} MB | Temp: {temp}°C")
            await asyncio.sleep(15)

await run_worker()
